# Bell examples (v0.10)

**Author:** Kolokolchikov Sergey Dmitrievich (`sergey.bell13@gmail.com`)

This notebook runs every numbered example of the Julia package Bell.
It is not a product release (`Project.toml` version `0.10.0`).

Open this file in Jupyter with the working directory `examples/`.
The first cell activates the examples environment (`examples/Project.toml`),
where Bell is a dev dependency on the repository root.

## What is an example, and what is not

| File | Role |
|------|------|
| `10`–`18` | Program examples. They load Bell and a structure file. |
| `common/print_util.jl` | Helper: `banner` / `section` for `10`–`18`. |
| `common/reference_values.jl` | Helper: OptiM path and COSY numbers for comparison tables. |

Each example is `run_example("NN_name/run.jl")` in a fresh module, so
constants of different examples do not clash. The printed output is the
same as `julia --project=examples examples/NN_name/run.jl`.

## Blocks of the program vs examples

| `src/` block | Example that uses it |
|--------------|----------------------|
| `coordinates.jl` | `15` (main), also `10`, `14` |
| `elements.jl` | `10`, `17` |
| `structure.jl` | `10`–`16`, `18`; FODO cell below (no OptiM) |
| `hamiltonians.jl` | `11`, `17` |
| `lie_algebra.jl` | `11`, `17` |
| `differential_algebra.jl` | `11`, `12` (check only) |
| `linear.jl` | `10` |
| `higher_order.jl` | `12` |
| `normal_form.jl` | `18` |
| `tracking.jl` | `11` |
| `exact_integrator.jl` | `12`, `14` |
| `longitudinal.jl` | `12`, `14` |
| `acceleration.jl` | `14` |
| `optimizers.jl` | `13` |
| `plotting.jl` | `10`, `12`, `13`, `14` |
| `symplectic.jl` | `10`, `11`, `15` |
| `optim_converter.jl` | `10` and every script that calls `convert_optim` |

`16` compares an external BLonD `(dt, dE)` drift with Bell tracking.
Those BLonD formulae are not a `src/` block.


In [1]:
import Pkg
# Jupyter cwd is the notebook folder (examples/).
Pkg.activate(pwd())
Pkg.instantiate()
ENV["GKSwstype"] = "100"   # headless Plots
using Dates

# Each example file is a script for Main.  A fresh module avoids const
# clashes between examples but must still have include().
function run_example(path)
    m = Module()
    Core.eval(m, :(include(x) = Base.include(@__MODULE__, x)))
    Base.include(m, path)
    return nothing
end

println("cwd      = ", pwd())
println("project  = ", Base.active_project())
println("started  = ", now())


  Activating project at `C:\GIT_REPOS\BELL\examples`


cwd      = C:\GIT_REPOS\BELL\examples
project  = C:\GIT_REPOS\BELL\examples\Project.toml
started  = 2026-10-30T13:35:16.033

## FODO through Bell (`lattices/fodo.bell`)

This cell loads a structure file directly:
`load_structure` → `Lattice` → one-turn `R` → closed Twiss.
It is the structure-file block without OptiM.


In [2]:
using Bell, Printf
lat = load_structure(joinpath(pwd(), "lattices", "fodo.bell"))
C = circumference(lat)
po = periodic_optics(ring_matrix(lat), lat.ref, C)
@printf("  name      = %s\n", lat.name)
@printf("  elements  = %d\n", length(lat.elements))
@printf("  C         = %.6f m\n", C)
@printf("  Qx        = %.6f\n", po.x.mu / 2pi)
@printf("  Qy        = %.6f\n", po.y.mu / 2pi)
@printf("  alpha_c   = %.8f\n", po.alpha_c)
@printf("  CLOSED check uses linear.jl + structure.jl + hamiltonians.jl\n")


  name      = fodo
  elements  = 128
  C         = 188.800000 m
  Qx        = 0.955869
  Qy        = 0.798840
  alpha_c   = 0.03888109


  CLOSED check uses linear.jl + structure.jl + hamiltonians.jl


## 10 — Linear optics of `magnetic_2`

OptiM `.opt` is converted to `lattices/magnetic_2.bell`. Bell loads the
structure file. Closed Twiss, dispersion, tunes, comparison with OptiM
and COSY numbers.

Blocks: converter, structure, coordinates, elements, Hamiltonians,
linear, symplectic, plotting.


In [3]:
run_example(joinpath("10_linear_optics", "run.jl"))



  Linear optics of magnetic_2


--- 1. Lattice file



  file                 : C:\GIT_REPOS\BELL\examples\common\..\lattices\magnetic_2.opt
  header               : Energy = 270.0 MeV (kinetic), Mass = 1875.612945 MeV
  sequence tokens      : 84, 'Number of periods' in file = 1, repetitions used = 1

--- 2. Reference particle

particle            : deuteron (Z = 1.0)
rest energy used    : 1875.61294257 MeV
kinetic energy      : 270.0 MeV
gamma0, beta0       : 1.14395294, 0.48563473
p0*c                : 1041.98416 MeV
B*rho               : 3.475685 T*m
1/gamma0^2          : 0.76415891
  (physical deuteron mass would be 1875.61294257 MeV -> B*rho = 3.4756850 T*m; not used)

--- 3. Elements

  OS2   x8   Drift(OS2, L=0.06 m)


  QF    x16  Quadrupole(QF, L=0.21 m, k1=0.5212441755015996 1/m^2)
  BM    x16  SBend(BM, L=5.329414799999999 m, angle=0.3926990753258494 rad, k1=0.0)
  QD    x16  Quadrupole(QD, L=0.21 m, k1=-0.5805856428570646 1/m^2)
  OS1   x6   Drift(OS1, L=0.12 m)
  SF2   x4   Sextupole(SF2, L=0.12 m, k2=0.0 1/m^3)
  SD    x4   Sextupole(SD, L=0.12 m, k2=0.0 1/m^3)
  SF1   x2   Sextupole(SF1, L=0.12 m, k2=0.0 1/m^3)
  QF1   x4   Quadrupole(QF1, L=0.21 m, k1=0.5565959259588208 1/m^2)
  OBM   x4   Drift(OBM, L=5.329414799999999 m)
  QD1   x4   Quadrupole(QD1, L=0.21 m, k1=-0.5565959259588208 1/m^2)

  number of elements    : 84
  circumference C       : 117.3882960 m
  total bending angle   : 359.999994158 deg


  dipoles: 16, quads: 40, sextupoles: 10

--- 4. Element linear maps

  R = exp(L S A), A = Hessian of H at z=0; symplectic error |R'SR - S|.



  OS2    R11= 1.000000 R12= 0.060000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.045850  sympl.err=0.0e+00
  QF     R11= 0.988529 R12= 0.209196 R21=-0.109042 R33= 1.011515 R16= 0.000000 R56= 0.160473  sympl.err=2.2e-16
  BM     R11= 0.923880 R12= 5.193490 R21=-0.028198 R33= 1.000000 R16= 1.033049 R56= 3.936595  sympl.err=6.7e-16
  QD     R11= 1.012829 R12= 0.210897 R21= 0.122444 R33= 0.987225 R16= 0.000000 R56= 0.160473  sympl.err=3.3e-16
  OS1    R11= 1.000000 R12= 0.120000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.091699  sympl.err=0.0e+00


  SF2    R11= 1.000000 R12= 0.120000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.091699  sympl.err=0.0e+00
  SD     R11= 1.000000 R12= 0.120000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.091699  sympl.err=0.0e+00
  SF1    R11= 1.000000 R12= 0.120000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.091699  sympl.err=0.0e+00
  QF1    R11= 0.987752 R12= 0.209142 R21=-0.116408 R33= 1.012298 R16= 0.000000 R56= 0.160473  sympl.err=2.2e-16
  OBM    R11= 1.000000 R12= 5.329415 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 4.072520  sympl.err=4.4e-16
  QD1    R11= 1.012298 R12= 0.210860 R21= 0.117364 R33= 0.987752 R16= 0.000000 R56= 0.160473  sympl.err=2.2e-16

--- 5. One-turn matrix and closed optics

  one-turn 6x6 matrix (canonical x, px, y, py, l, delta):
    

... (3 outputs left out)

  normal-form check: |A^-1 R A - rotation| = 4.73e-14

--- 6. Optics along the ring + CLOSED check



  rel. beta_x 2.2e-14, alpha_x 4.9e-15, rel. beta_y 1.1e-14, alpha_y 2.1e-14, D 9.9e-13, D' 5.5e-14, Qfrac 5.7e-15/1.1e-14
  CLOSED: yes (all closure errors < 1e-8)

  tunes Qx = 2.46172014   Qy = 2.46133732  (integer from accumulated phase advance)
  max beta_x = 19.3677 m, max beta_y = 19.3646 m, max D = 11.7016 m, min D = 0.0068 m

  alpha_c from (1/C) int h*D ds (trapezoid)  = 0.23047037  (independent check)

--- 7. Plot

  written: 

C:\GIT_REPOS\BELL\examples\10_linear_optics\output\magnetic_2_twiss.png

--- 8. Comparison with OptiM / COSY

  quantity                         Bell          reference         diff  source


  beta_x(s=0) [m]           19.36752242            19.3675    2.242e-05  OptiM Initial
  beta_y(s=0) [m]           4.054470469            4.05447    4.690e-07  OptiM Initial
  alpha_x(s=0)         -0.0002519089034       -0.000251909    9.663e-11  OptiM Initial
  alpha_y(s=0)         -0.0001881504894       -0.000188151    5.106e-10  OptiM Initial
  D_x(s=0) [m]            0.01109398438                  0    1.109e-02  OptiM Initial
  D'_x(s=0)             -0.000506160301        -0.00050616   -3.010e-10  OptiM Initial
  Qx fractional            0.4617201386       0.4617201484   -9.739e-09  COSY wp_natural.dat
  Qy fractional             0.461337323       0.4613373213    1.772e-09  COSY wp_natural.dat
  alpha_c                  0.2304702783       0.2304702825   -4.212e-09  COSY integrals_Icirc.dat

Done.


## 11 — Lie maps and tracking

Element `H`, Dragt–Finn `f3`, ring Lie product versus the independent
RK4 Taylor map, chromaticity, symplectic factor-by-factor tracking.

Blocks: Hamiltonians, Lie algebra, differential algebra (check),
tracking, symplectic.


In [4]:
run_example(joinpath("11_lie_maps", "run.jl"))



  2nd-order Lie maps, symplectic tracking, Taylor-map check

  test amplitude z_amp (x, px, y, py, l, delta) = 

[0.003, 0.0002, 0.002, 0.0002, 0.001, 0.002]

--- 1. Elements: H, Lie map, f3, symplecticity

  Lie map = exp(-L:H:) z to order 2; f3 = Dragt-Finn cubic generator.

  elem  |lin-expLSA|  recon.err   TM sym@0 TM sym@amp LF sym@amp  nterms f3


  OS2      6.9e-18    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3


  QF       2.2e-16    2.8e-17    1.1e-16    1.6e-12    2.2e-16          7
  BM       2.7e-15    1.8e-15    4.4e-16    2.5e-09    6.7e-16         13


  QD       2.2e-16    2.8e-17    2.2e-16    1.9e-12    1.1e-16          7
  OS1      1.4e-17    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3
  SF2      1.4e-17    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3
  SD       1.4e-17    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3


  SF1      1.4e-17    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3
  QF1      2.2e-16    5.6e-17    4.4e-16    1.8e-12    6.7e-16          7
  OBM      8.9e-16    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3
  QD1      2.2e-16    5.6e-17    4.4e-16    1.8e-12    8.9e-16          7


... (6 outputs left out)

  one turn: max |Lie tracking - Taylor map| = 4.73e-10 (expected O(|z|^3))


  symplectic error of 1-turn tracking Jacobian at z0: 9.77e-15 (Taylor map: 1.28e-06)



  1000 turns in 8.4 s; action Jx: mean 1.0327e-07, rel. spread 2.37e-03; Jy: mean 1.2317e-07, rel. spread 2.71e-03
  (spread is due to the nonlinear (2nd-order) terms and d-dependence of Twiss, not to loss of symplecticity)
  final delta = 1.000000000000000e-03 (must equal start 1.000000000000000e-03: magnetic lattice)

--- 5. Comparison with COSY output numbers

  quantity               Bell           COSY         diff  source


  xi_x            -1.46153575    -1.46153571   -3.961e-08  COSY OV 3 2 1 + TP, FR 0 (cosy_ov321.txt xi_x_p)
  xi_y            -2.22548498    -2.22548493   -4.509e-08  COSY OV 3 2 1 + TP, FR 0 (cosy_ov321.txt xi_y_p)
  alpha_0          0.23047028     0.23047028   -4.200e-09  COSY OV 3 3 0, FR 0 (ov330_map_analysis.json alpha_c)

Done.


## 12 — Higher order (`D2`, `η`, `ξ`)

Second-order dispersion, momentum compaction, slip, chromaticity.
Cross-check with the exact-H integrator.

Blocks: higher order, differential algebra, exact integrator,
longitudinal (tune formula), plotting.


In [5]:
run_example(joinpath("12_higher_order", "run.jl"))



  Second-order dispersion, compaction, slip, chromaticity

  lattice magnetic_2, C = 117.388296 m, E_kin = 270.0 MeV, gamma0 = 1.14395294, beta0 = 0.48563473

--- 1. One-turn Taylor map (order 3)

  Element maps by RK4 in truncated polynomials, composed.

  built in 0.0 s



--- 2. Second-order dispersion

  (I-R4) D2 = Q4(D1,0,1): delta^2 part of the closed orbit.

  s=0:  D =  0.01109398 m   D' = -5.06160279e-04   D2 = -10.05493202 m   D2' =  4.59065838e-01


  along ring (element exits): max D = 11.70161 m, max|D2| = 14.40743 m, D2 range [-10.08251, 14.40743] m
  closure: D2(C) - D2(0) = 6.06e-09, D2'(C) - D2'(0) = 2.12e-10



  vs RK4 element maps: max |D2 Lie minus RK4| = 6.06e-09

--- 3. Momentum compaction and slip factor to 2nd order

  dC/C = alpha0 d + alpha1 d^2 ;  dT/T0 = eta0 d + eta1 d^2.

  l1 = (l|d)_closed = 62.64879908 m,  l2 = (l|d^2)_closed = -71.52912795 m
  v/v0 - 1 = 0.76415891 d + -0.27033011 d^2  (check vs exact at d=1e-3: 1.62e-11)

  alpha0 = 0.23047028   alpha1 = 0.33900771   gamma_t = 2.083016
  eta0   = -0.53368863   eta1   = 1.01716074   (identity formula: 1.01716074, diff 0.0e+00)
  delta_t (2nd fixed point of longitudinal motion, alpha-bucket) ~ -eta0/eta1 = 0.5247



  observables (Lie factor): eta0 = -0.53368863  eta1 = 1.01716074  (Taylor map eta0 = -0.53368863, eta1 = 1.01716074)
  observables Twiss from F.R: Qx = 0.46172014  Qy = 0.46133732  alpha_c = 0.23047028

--- 4. Cross-check by exact-H tracking

  Closed orbit at fixed delta (Newton), dC/C from l, quadratic fit.

  fit x_co = a d + b d^2 + c d^3:    a = 0.01108484 (map D 0.01109398),  b = -10.054867 (map D2 -10.054932)


  fit dC/C = a d + b d^2 + c d^3:    a = 0.23047050 (map alpha0 0.23047028), b = 0.339007 (map alpha1 0.339008)

--- 5. Chromaticity, first and second order

  Map: Q(d) from the Jacobian on the off-momentum fixed point.
  Tracking: exact H, 256 turns of 1e-5 m oscillation, linear-tune fit.

  map:  Qx0 = 0.46172014  xi1_x = -1.461538  xi2_x = d2Qx/dd2 = 8.5547
        Qy0 = 0.46133732  xi1_y = -2.225488  xi2_y = d2Qy/dd2 = 8.1041



  tracking fit: Qx0 = 0.46171903  xi1_x = -1.461531  xi2_x = 8.5546
                Qy0 = 0.46133566  xi1_y = -2.225479  xi2_y = 8.1038

  delta         Qx track        Qx map   Qx map-poly      Qy track        Qy map


  -1.5e-03   0.463920968   0.463922712   0.463922069   0.464683019   0.464685775
  -1.0e-03   0.463184845   0.463186139   0.463185953   0.463565198   0.463567178
  -5.0e-04   0.462450868   0.462451999   0.462451977   0.462449414   0.462451117
  0.0e+00    0.461719032   0.461720139   0.461720139   0.461335661   0.461337323
  5.0e-04    0.460989335   0.460990418   0.460990439   0.460223934   0.460225557
  1.0e-03    0.460261774   0.460262707   0.460262878   0.459114227   0.459115605
  1.5e-03    0.459536344   0.459536884   0.459537456   0.458006537   0.458007273



  written: C:\GIT_REPOS\BELL\examples\12_higher_order\output\chromaticity_natural.png

--- 6. Longitudinal motion from eta0, eta1

turn-by-turn longitudinal motion (n = turn number, phi = -h w0 dt, delta = dp/p):
  dphi/dn   = -2 pi h (eta0 delta + eta1 delta^2)
  ddelta/dn = (Z V T/(beta^2 E)) (sin(phi_s + phi) - sin(phi_s))
  H(phi,delta) = -2 pi h (eta0 delta^2/2 + eta1 delta^3/3)
                 + (Z V T/(beta^2 E)) (cos(phi_s + phi) + phi sin(phi_s))
  small amplitude:  Q_s = sqrt( h eta0 Z V T cos(phi_s) / (2 pi beta^2 E) ),
  stable iff eta0*cos(phi_s) > 0.  eta1 adds a second fixed point at
  delta ~ -eta0/eta1 (alpha-bucket) and makes the bucket asymmetric.
  example at 270 MeV, h=5, V=0.2 MV, T=1, phi_s=180 deg: Q_s = 0.012956

--- 7. Exact integrator vs Lie tracking vs linear map (one turn, no RF)



  |exact - Lie(order 3)| = 2.36e-08,  |exact - Taylor map| = 2.35e-08,  |exact - linear| = 6.90e-05


  symplectic error of exact-integrator one-turn Jacobian at z0: 3.22e-14

--- 8. Notes

  Natural lattice (no fringe fields): compare with COSY FR 0 below.
  sextupole strengths in magnetic_2.opt are all zero (natural lattice above).

  COSY natural chromaticity (reference only): xi_x = -1.461536, xi_y = -2.225485

Done.


## 13 — Optional sextupole families

Places `SXD` / `SXB` / `SYB` and solves two example targets. Other
examples leave sextupoles at zero.

Block: optimizers (optional; not part of the model).


In [6]:
run_example(joinpath("13_sextupole_optimization", "run.jl"))



  Sextupole families and example optimisation


--- 1. Placement

  Maxima of D_x, beta_x, beta_y with D_x > 0.5 m.



  fam  max of      s_max     value slot     s_slot   beta_x   beta_y      D_x


  SXD  D_x        23.418   11.7016 SF1      23.478   19.368    4.055   11.702
  SXD  D_x        82.112   11.7016 SF1      82.172   19.368    4.055   11.702
  SXB  beta_x     11.679   16.4587 SF2      11.739   16.459    3.325    5.832
  SXB  beta_x     35.156   16.4586 SF2      35.216   16.459    3.325    5.871
  SXB  beta_x     70.373   16.4587 SF2      70.433   16.459    3.325    5.832
  SXB  beta_x     93.851   16.4586 SF2      93.911   16.459    3.325    5.871
  SYB  beta_y      5.799   18.7847 OS1       5.869    3.663   18.785    1.123
  SYB  beta_y     17.678   18.7811 SD       17.608    3.662   18.781    4.927
  SYB  beta_y     29.277   18.7811 SD       29.347    3.662   18.781    4.927
  SYB  beta_y     41.156   18.7847 OS1      41.086    3.663   18.785    1.123
  SYB  beta_y     64.494   18.7847 OS1      64.564    3.663   18.785    1.123
  SYB  beta_y     76.372   18.7811 SD       76.302    3.662   18.781    4.927
  SYB  beta_y     87.971   18.7811 SD       88.041    3.662   18

  (computed in 1.4 s)

  family         dxi1_x       dxi1_y      dalpha1        deta1   dD2(0) [m]
  SXD           4.32833     -0.90618     -1.63792     -1.63792     89.86468
  SXB           3.67887     -0.74301     -0.40970     -0.40970     -0.03441
  SYB           0.84330     -4.33701     -0.24240     -0.24240      9.45545

--- 3. Solutions

  A: xi1_x = xi1_y = alpha1 = 0
  B: xi1_x = xi1_y = 0, min rms D2

  solution A: 

SXD: k2 = 0.230414 m^-3 (k2*L = 0.02765 m^-2, B'' = 0.80085 T/m^2);  SXB: k2 = 0.265267 m^-3 (k2*L = 0.031832 m^-2, B'' = 0.92199 T/m^2);  SYB: k2 = -0.606726 m^-3 (k2*L = -0.072807 m^-2, B'' = -2.1088 T/m^2)
  solution B: SXD: k2 = 0.145559 m^-3 (k2*L = 0.017467 m^-2, B'' = 0.50592 T/m^2);  SXB: k2 = 0.364953 m^-3 (k2*L = 0.043794 m^-2, B'' = 1.2685 T/m^2);  SYB: k2 = -0.606075 m^-3 (k2*L = -0.072729 m^-2, B'' = -2.1065 T/m^2)



--- 4. Before / after

  All from the order-3 one-turn map; xi2 = d2Q/ddelta2.

  quantity              natural  A (xi,alpha1)   B (xi,D2rms)


  Qx frac              0.461720       0.461720       0.461720
  Qy frac              0.461337       0.461337       0.461337
  xi1_x               -1.461538       0.000005       0.000003
  xi1_y               -2.225488      -0.000007      -0.000007
  xi2_x                8.554735       3.934367       5.164929
  xi2_y                8.104123      -0.717107       1.235665
  D(0) [m]             0.011094       0.011094       0.011094
  D2(0) [m]          -10.054932       4.905161      -2.717586
  max|D2| [m]         14.407434      10.037720       3.809482
  rms D2 [m]           7.558396       5.728021       2.411268
  alpha0               0.230470       0.230470       0.230470
  alpha1               0.339008       0.000000       0.097987
  eta0                -0.533689      -0.533689      -0.533689
  eta1                 1.017161       0.678153       0.776140

  (linear-model prediction check A: xi1_x, xi1_y, alpha1 should be ~0)

--- 5. Tracking check of solution A

  tracking fit A: xi1_

  map         A: xi1_x = 0.00000  xi2_x = 3.934   xi1_y = -0.00001  xi2_y = -0.717



  written: C:\GIT_REPOS\BELL\examples\13_sextupole_optimization\output\chromaticity_corrected.png
  written: C:\GIT_REPOS\BELL\examples\13_sextupole_optimization\output\sextupoles_dispersion2.png

Done.


## 14 — Finite RF cavity and acceleration

Cavity checks, synchrotron tune, ramp 10 → 270 MeV, adiabatic damping.
Injection energy 10 MeV is assumed (not in the OptiM file). Takes a
few minutes.

Blocks: exact integrator, acceleration, longitudinal, plotting.


In [7]:
run_example(joinpath("14_rf_acceleration", "run.jl"))



  Finite RF cavity, synchrotron motion, acceleration


--- 1. Parameters



  injection E_kin = 10.0 MeV (ASSUMED, 5 MeV/u; not given in magnetic_2.opt)
  final     E_kin = 270.0 MeV (135 MeV/u)

  cavity: L = 1.00 m in the first OBM drift (element 37)


          V = 0.200 MV, h = 5, phi_s = 150.0 deg
  C = 117.388296 m (unchanged)
  transit factor T = sin(th/2)/(th/2), th = 2 pi h L/C = 0.26762 rad -> T = 0.997018



  injection: beta = 0.102852 gamma = 1.005332 p0 = 193.9388 MeV/c Brho = 0.64691 T m  f0 = 262.6685 kHz  f_RF = 1.3133 MHz
  final    : beta = 0.485635 gamma = 1.143953 p0 = 1041.9842 MeV/c Brho = 3.47569 T m  f0 = 1240.2397 kHz  f_RF = 6.2012 MHz

  energy gain per turn Z V T sin(phi_s) = 0.099702 MeV -> about 2608 turns for the ramp

--- 2. Cavity checks

  Single passes through the cavity at 270 MeV (no renormalisation).



  synchronous particle: delta_out = 1.9701321011e-04
                       dE = 0.099700857 MeV   vs Z V T sin(phi_s) = 0.099701840 MeV

  thin limit L = 0.1    m: delta_out = 3.802214653167e-04  (thin kick 3.802277730094e-04, diff -6.31e-09)
  thin limit L = 0.01   m: delta_out = 3.802276257953e-04  (thin kick 3.802277730094e-04, diff -1.47e-10)
  thin limit L = 0.001  m: delta_out = 3.802277631235e-04  (thin kick 3.802277730094e-04, diff -9.89e-12)



  symplectic error of cavity map Jacobian: 2.22e-16

--- 3. Synchrotron tune at fixed energy (phi_s = 180 deg)

  Q_s = sqrt(h eta0 Z V T cos(phi_s) / (2 pi beta^2 E)), eta0 = alpha0 - 1/gamma^2.



... (10 outputs left out)

  (a) C:\GIT_REPOS\BELL\examples\14_rf_acceleration\output\longitudinal_ramp.gif
      every 20th turn; left: absolute, right: delta / current bucket height


[ Info: Saved animation to C:\GIT_REPOS\BELL\examples\14_rf_acceleration\output\longitudinal_ramp.gif


  (b) C:\GIT_REPOS\BELL\examples\14_rf_acceleration\output\longitudinal_elements.gif
      one turn at injection, frame = exit of each element

Done.


[ Info: Saved animation to C:\GIT_REPOS\BELL\examples\14_rf_acceleration\output\longitudinal_elements.gif


## 15 — User ↔ canonical coordinates

Exact nonlinear conversion `(x, x', y, y', φ, δ) ↔ (x, px, y, py, l, δ)`.
The user Jacobian is not symplectic.

Block: coordinates.


In [8]:
run_example(joinpath("15_coordinates", "run.jl"))



  Exact coordinate transformation user <-> canonical


--- 1. Formulas

  canonical -> user: x' = px/ps, y' = py/ps, ps = sqrt((1+delta)^2 - px^2 - py^2)
                     phi = 2 pi h l / (C v(delta)/v0)
  user -> canonical: px = x'(1+delta)/sqrt(1+x'^2+y'^2), py likewise,
                     l = phi C (v/v0) / (2 pi h)
  Both functions are generic: on TaylorN they give the full expansion.

  x'(px,py,delta) to order 3 : 1.0*px - 1.0*px*delta + 0.5*px^3 + 0.5*px*py^2 + 1.0*px*delta^2
  phi(l,delta)    to order 3 : 0.267624*l - 0.204507*l*delta + 0.228623*l*delta^2



  largest nonlinear coefficient in T: 1.500e+00  (> 0 => conversion is nonlinear)

--- 2. Polynomial round trip T(T^-1(u)) - u

  component 1: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00


  component 2: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00
  component 3: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00
  component 4: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00
  component 5: max|coef| orders <= 3: 1.11e-16   orders > 3 (kept up to JET_ORDER): 0.00e+00
  component 6: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00

--- 3. Point residuals at large amplitude

  u = (5 mm, 20 mrad, 5 mm, 20 mrad, 1.0 rad, 1e-2)

  exact functions round trip     : max|u - T(T^-1 u)| = 0.00e+00
  polynomial T (order 3) vs exact T at that point: max diff = 8.73e-07 (truncation error, O(amp^4))


  compare: linearised T would give max diff = 7.61e-03

--- 4. One-turn map coefficients in both systems (270 MeV, RF off)



  coefficient                             canonical             user
  x | px  vs x | x'                  4.61348267e+00   4.61348267e+00
  x | delta                          2.42044478e-02   2.42044478e-02
  px | delta  vs  x' | delta        -8.61271681e-04  -8.61271681e-04
  l | delta  vs  phi | delta         6.26488018e+01   1.67663236e+01
  l | delta^2  vs phi | delta^2     -7.15364200e+01  -3.19569992e+01
  l | px^2  vs  phi | x'^2          -8.62567312e+01  -2.30843723e+01
  px | px delta vs x'|x' delta       2.13576203e+00   2.13576203e+00
  x | px^2 delta vs x|x'^2 delta     3.43450442e+02   3.43541751e+02

  (phi coefficients include the factor 2 pi h / C and the v(delta) terms)

--- 5. Symplecticity

  max |J^T S J - S| of the map Jacobian at z = (1e-3, 1e-3, 1e-3, 1e-3, 0, 1e-3)



  canonical map: 6.75e-06  (zero up to truncation at order 3)
  user map     : 1.19e-01  (not small: (x,x',y,y',phi,delta) is not a canonical set)

Done.


## 16 — BLonD-like longitudinal drift (external formulae)

`(dt, dE)` drift of BLonD compared with element tracking. The formulae
live in this example, not in `src/`.


In [9]:
run_example(joinpath("16_blond_longitudinal", "run.jl"))



  BLonD-like longitudinal drift vs element tracking

  mass used = 1875.61294257 MeV (CODATA deuteron); OptiM header Mass = 1875.613 MeV (ignored)

--- 1. Momentum compaction from the ring map (BLonD alpha_0,1,2)

  Taylor map in 0.0 s



  path_expansion (order 2):  alpha0 = 0.23047028  alpha1 = 0.33900771
  blond_alphas fit:   alpha0 = 0.23047028  alpha1 = 0.33901173  alpha2 = -0.14921926
  BLonD eta (eq.15):         eta0 = -0.53368863  eta1 = 0.73234120  eta2 = -0.15322364
  path_expansion eta:        eta0 = -0.53368863  eta1 = 1.01716074  (TOF identity)
  |alpha0 fit - path| = -1.20e-11  |alpha1 fit - path| = 4.02e-06

--- 2. One-turn magnetic drift: element tracking vs BLonD

  Particles on the closed orbit at fixed delta (x = D delta, ...). No RF.

       delta     Dphi exact     Dphi BLonD    Dphi simple    rel BLonD   rel simple
  -2.0000e-03  -3.366068e-02  -3.366074e-02  -3.350701e-02     1.78e-06    -4.57e-03


  -1.0000e-03  -1.679828e-02  -1.679831e-02  -1.675992e-02     2.00e-06    -2.28e-03
  -5.0000e-04  -8.391137e-03  -8.391154e-03  -8.381560e-03     2.07e-06    -1.14e-03
  5.0000e-04   8.375159e-03   8.375177e-03   8.384763e-03     2.10e-06     1.15e-03
  1.0000e-03   1.673437e-02   1.673440e-02   1.677273e-02     2.06e-06     2.29e-03
  2.0000e-03   3.340504e-02   3.340510e-02   3.355826e-02     1.86e-06     4.59e-03

--- 3. With RF: thin kick + BLonD exact drift vs full tracking (32 turns)

  Stationary bucket phi_s = 180 deg, V = 0.2 MV.
  BLonD uses an instantaneous kick; tracking uses the finite cavity.

    turn    phi track    phi BLonD      d track      d BLonD


       1      0.01671      0.01673  9.97414e-04  1.00000e-03
       4      0.06573      0.06584  9.50526e-04  9.60574e-04
       8      0.12461      0.12479  8.01874e-04  8.20581e-04
      16      0.19865      0.19879  2.79107e-04  3.07810e-04
      32      0.10752      0.10685 -8.47817e-04 -8.33517e-04

  max |phi_track - phi_BLonD| over 32 turns = 6.732e-04 rad
  max |delta_track - delta_BLonD|           = 2.938e-05
  (small residual expected: finite cavity length + transverse coupling vs thin kick + 1D drift)

Done.


## 17 — Lie product of two factors

`concatenate(Fa, Fb)` versus the Lie series of a pair of elements.
Fixes the sign of the BCH term.

Block: Lie algebra (product of Dragt–Finn factors).


In [10]:
run_example(joinpath("17_lie_product", "run.jl"))



  Lie product of Dragt-Finn factors to degree 4


--- 1. Single element: Lie series vs Dragt-Finn reconstruction

  elem          recon         |f3|         |f4|
  D         0.000e+00    1.000e+00    1.000e+00


  QF        5.551e-17    2.032e-01    2.049e-01
  QD        5.551e-17    2.032e-01    2.049e-01
  SX        2.220e-16    8.000e-01    1.000e-01
  B         2.220e-16    7.493e-01    7.503e-01



--- 2. Pair of elements: concatenate vs composed Lie series

  pair                |Δf3|        |Δf4|    |Δf4 opp|         |ΔR|       |Δmap|


  D+QF            2.220e-16    2.220e-16    5.653e-02    0.000e+00    4.441e-16
  QF+QD           5.551e-17    5.551e-17    1.030e-05    2.220e-16    3.331e-16


  QF+SX           2.220e-16    1.110e-16    6.735e-01    1.110e-16    5.551e-16
  SX+D            2.776e-17    4.510e-17    3.200e+00    0.000e+00    4.441e-16


  B+QF            1.110e-16    1.110e-16    4.191e-02    1.388e-17    2.220e-16
  QF+B            1.110e-16    1.110e-16    3.713e-02    1.388e-17    4.441e-16


  SX+SX           5.551e-17    1.110e-16    2.560e-01    0.000e+00    4.441e-16

--- 3. Three elements (fold)



  |Δf3| = 3.331e-16  |Δf4| = 1.110e-16  |Δmap| = 5.551e-16

Done.


## 18 — Normal form on generators

One-turn Lie factor with thin RF, then the homological equation on
`f3`, `f4`. Torus average of laboratory `δ`.

Block: normal form.


In [11]:
run_example(joinpath("18_lie_normal_form", "run.jl"))



  One-turn Lie factor and normal form

  one-turn factor in 0.17 s; |f3|=8.626e+01  |f4|=3.529e+02


  |const of factor map| = 0.000e+00 (origin must stay)



  NF in 2.28 s
  Q = (0.461720, 0.461337, 0.012960)
  |A^{-1} R A - rot| = 4.663e-15   |A'SA - S| = 7.134e-16

  |f3| before = 2.565e-01   after F3 = 4.441e-16
  |f4 - h4| after F4 = 1.271e-13
  d (torus, from T) = 1.46580656e-01  2.23198761e-01  2.47854827e-03
  d (grid check)    = 1.46580656e-01  2.23198761e-01  2.47854827e-03
  |d - d_grid|      = 7.131e-15

  d (TOF formula)   = 1.46580802e-01  2.23199062e-01  2.47854844e-03


  rel d − formula   = -9.965e-07  -1.353e-06  -6.825e-08

  cubic homological residual |f3+F3-F3∘r| = 4.441e-16

Done.


## Done

Every numbered example has been included. Plots and GIFs are under
`examples/NN_name/output/`.


In [12]:
println("finished = ", now())


finished = 2026-10-30T13:42:49.732
